In [62]:
import numpy as np
import pandas as pd
N = 50000
np.random.seed(42)
col1 = np.random.standard_normal(N)
col2 = np.random.uniform(-5, 5, N)
col3, col4, col5 = np.random.multivariate_normal([1, -1, 2], [[1.0, 0.8, -0.3], [0.8, 1.0, 0.1], [-0.3, 0.1, 1.0]], N).reshape(3, -1)
col6 = np.random.rand(N)
col6 = np.random.choice(['Copper', *2*['Aluminium'], *3*['Titanium'], *4*['Tungsten']], N)
col7 = 2*col2**2 - 3*col2 + 1
col8rand = np.random.rand(N)
col8 = [np.random.normal(-3, 1) if rnd <= 0.7 else np.random.normal(3, 1) for rnd in col8rand]
col9 = [col1[i] * col2[i] if col6[i] != 'Copper' else None for i in range(N)]
df = pd.DataFrame({'col1': col1,'col2': col2,'col3': col3, 'col4': col4, 'col5': col5, 'col6': col6, 'col7': col7, 'col8': col8, 'col9': col9})
df

,col1,col2,col3,col4,col5,col6,col7,col8,col9
0,0.496714,-4.972295,0.856206,0.449057,-2.792878,Tungsten,65.364319,-3.982535,-2.469809
1,-0.138264,-1.174774,-1.104457,2.022651,0.897015,Tungsten,7.284507,-1.865070,0.162429
2,0.647689,4.140493,0.364906,-0.468112,0.758790,Aluminium,22.865882,-3.263226,2.681750
3,1.523030,4.477549,0.820167,2.298262,-1.647135,Copper,27.664247,2.511258,NaN
4,-0.234153,-4.464408,-0.931179,-0.346679,0.579473,Titanium,54.255091,-3.139977,1.045356
...,...,...,...,...,...,...,...,...,...
49995,0.056799,1.641270,1.613612,3.560890,-0.230378,Titanium,1.463726,-2.984232,0.093222
49996,-0.024923,-4.471092,-0.611411,0.536867,3.004438,Titanium,54.394594,-0.969403,0.111432
49997,0.500085,3.481545,1.098155,-2.128485,2.455626,Aluminium,14.797675,-2.538976,1.741068
49998,0.265215,4.167589,1.852977,1.266711,0.255052,Tungsten,23.234829,-2.824879,1.105309


In [63]:
y = pd.read_csv('train_data.csv')

In [64]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.impute import SimpleImputer
import optuna
df = pd.get_dummies(df)
imputer = SimpleImputer()
df = imputer.fit_transform(df)
train_x, val_x, train_y, val_y = train_test_split(df, y, test_size=0.05)
len(train_x), len(val_x)

(47500, 2500)

In [65]:
# sgd_model = SGDRegressor()
# sgd_model.fit(train_x, train_y)
# pred = sgd_model.predict(val_x)
# root_mean_squared_error(val_y, pred)

In [ ]:
def objective(trial):
    # max_depth = trial.suggest_int('max_depth', 2, 10)
    # n_estimators = trial.suggest_int('n_estimators', 50, 400)
    # lr = trial.suggest_float("lr", 1e-4, 0.3, log=True)
    # model = XGBRegressor(max_depth=max_depth, n_estimators=n_estimators, learning_rate=lr)
    l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
    max_iter = trial.suggest_int('max_iter', 50, 1000)
    model = SGDRegressor(l1_ratio=l1_ratio, max_iter=max_iter)
    model.fit(train_x, train_y)
    pred = model.predict(val_x)
    score = root_mean_squared_error(val_y, pred)
    return score

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, show_progress_bar=True)
study.best_value, study.best_params

[I 2026-05-15 17:11:37,621] A new study created in memory with name: no-name-695ee4f5-94d5-4538-8c51-ada7bfc674cf


  0%|          | 0/50 [00:00<?, ?it/s]

c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[I 2026-05-15 17:11:40,920] Trial 0 finished with value: 8.649204507514396 and parameters: {'l1_ratio': 0.1830643069675132, 'max_iter': 909}. Best is trial 0 with value: 8.649204507514396.


c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


...

c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[I 2026-05-15 17:13:52,951] Trial 44 finished with value: 71.51015554846747 and parameters: {'l1_ratio': 0.1786375662002834, 'max_iter': 636}. Best is trial 12 with value: 6.651656813113843.


c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[I 2026-05-15 17:13:55,846] Trial 45 finished with value: 98.56394747997882 and parameters: {'l1_ratio': 0.5408190962428353, 'max_iter': 736}. Best is trial 12 with value: 6.651656813113843.
[W 2026-05-15 17:13:57,183] Trial 46 failed with parameters: {'l1_ratio': 0.4377541900232842, 'max_iter': 380} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\raian\source\repos\AI\.env\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\raian\AppData\Local\Temp\ipykernel_2620\3454211827.py", line 9, in objective
    model.fit(train_x, train_y)
  File "c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\linear_model\_stochastic_gradi

KeyboardInterrupt: 